# Feature Engineering — MedRisk Readmission Prediction

This notebook walks through the clinical feature engineering pipeline that transforms raw EHR data into predictive features for 30-day readmission modeling.

In [ ]:
%matplotlib inline

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

## 1. Raw Data Overview

In [ ]:
try:
    from src.data.loader import load_raw_data
    df = load_raw_data()
    print("Loaded full dataset.")
except Exception:
    df = pd.read_csv("../data/sample/sample_data.csv", na_values=["?"])
    print("Full dataset not available — using sample data.")

print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())

In [ ]:
df[["diag_1", "diag_2", "diag_3"]].head(10)

## 2. ICD-9 Diagnosis Mapping

Raw ICD-9 codes have thousands of unique values. We map them to 9 clinically meaningful categories using the project's `map_icd9_to_category` function.

In [ ]:
from src.features.engineer import map_icd9_to_category, engineer_diagnosis_features

examples = ["250.13", "428", "786", "V45", "E819", "715", "162", np.nan]
print("ICD-9 Code -> Category Mapping Examples:")
print("-" * 40)
for code in examples:
    print(f"  {str(code):>10} -> {map_icd9_to_category(code)}")

In [ ]:
df_diag = engineer_diagnosis_features(df.copy())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(["diag1_category", "diag2_category", "diag3_category"]):
    order = df_diag[col].value_counts().index
    sns.countplot(data=df_diag, y=col, order=order, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} Distribution")
    axes[i].set_xlabel("Count")
plt.tight_layout()
plt.show()

## 3. Medication Feature Aggregation

The dataset contains 23 medication columns, each with values: **Up**, **Down**, **Steady**, or **No**. We aggregate these into three clinically interpretable features.

In [ ]:
from src.features.engineer import MEDICATION_COLUMNS, engineer_medication_features

print(f"Medication columns ({len(MEDICATION_COLUMNS)}):")
print(MEDICATION_COLUMNS)

present_cols = [c for c in MEDICATION_COLUMNS if c in df.columns]
print(f"\nValue distribution for 'insulin':")
print(df["insulin"].value_counts())

In [ ]:
df_med = engineer_medication_features(df.copy())

print("Aggregated medication features:")
print(f"  num_med_changes:  medications with dosage changed (Up or Down)")
print(f"  num_meds_active:  medications currently prescribed (Up, Down, or Steady)")
print(f"  insulin_changed:  binary flag if insulin dosage was modified")
print()
df_med[["num_med_changes", "num_meds_active", "insulin_changed"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df_med["num_med_changes"].hist(bins=15, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("num_med_changes")
df_med["num_meds_active"].hist(bins=15, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_title("num_meds_active")
df_med["insulin_changed"].value_counts().plot(kind="bar", ax=axes[2], color=["gray", "coral"])
axes[2].set_title("insulin_changed")
axes[2].set_xticklabels(["No (0)", "Yes (1)"], rotation=0)
plt.tight_layout()
plt.show()

## 4. Utilization Features

Prior healthcare utilization is a strong predictor of readmission. We combine inpatient, outpatient, and emergency visits into a total score and a binary high-utilizer flag.

In [ ]:
from src.features.engineer import engineer_utilization_features

df_util = engineer_utilization_features(df.copy())

print("Utilization features:")
print("  total_prior_visits = number_inpatient + number_outpatient + number_emergency")
print("  high_utilizer = 1 if total_prior_visits >= 5")
print()
df_util[["number_inpatient", "number_outpatient", "number_emergency", "total_prior_visits", "high_utilizer"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_util["total_prior_visits"].hist(bins=30, ax=axes[0], color="teal", edgecolor="white")
axes[0].set_title("Total Prior Visits Distribution")
axes[0].set_xlabel("total_prior_visits")
axes[0].axvline(5, color="red", linestyle="--", label="High utilizer threshold (5)")
axes[0].legend()

df_util["high_utilizer"].value_counts().plot(kind="bar", ax=axes[1], color=["gray", "red"])
axes[1].set_title("High Utilizer Flag")
axes[1].set_xticklabels(["No (0)", "Yes (1)"], rotation=0)
plt.tight_layout()
plt.show()

## 5. Age Ordinal Encoding

Age brackets are converted to ordinal integers for use in numeric models.

In [ ]:
from src.features.engineer import _AGE_BRACKET_MAP, engineer_age_ordinal

print("Age Bracket -> Ordinal Mapping:")
print("-" * 30)
for bracket, ordinal in sorted(_AGE_BRACKET_MAP.items(), key=lambda x: x[1]):
    print(f"  {bracket:>10} -> {ordinal}")

In [ ]:
df_age = engineer_age_ordinal(df.copy())

fig, ax = plt.subplots(figsize=(8, 4))
df_age["age_ordinal"].value_counts().sort_index().plot(kind="bar", ax=ax, color="mediumpurple")
ax.set_title("Age Ordinal Distribution")
ax.set_xlabel("Age Ordinal")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 6. New v2 Features

Additional intensity ratios and clinical flags that capture more nuanced patient characteristics.

In [ ]:
from src.features.engineer import run_feature_engineering

df_full = run_feature_engineering(df.copy())

# Compute v2 intensity features on engineered data
df_v2 = df_full.copy()
df_v2["lab_procedure_intensity"] = df_v2["num_lab_procedures"] / (df_v2["time_in_hospital"] + 1)
df_v2["procedure_intensity"] = df_v2["num_procedures"] / (df_v2["time_in_hospital"] + 1)
df_v2["medication_intensity"] = df_v2["num_medications"] / (df_v2["time_in_hospital"] + 1)
df_v2["diagnosis_complexity"] = df_v2["number_diagnoses"] / (df_v2["time_in_hospital"] + 1)
df_v2["med_change_ratio"] = df_v2["num_med_changes"] / (df_v2["num_meds_active"] + 1)

v2_features = [
    "lab_procedure_intensity", "procedure_intensity",
    "medication_intensity", "diagnosis_complexity", "med_change_ratio",
]
print("v2 Intensity Features:")
df_v2[v2_features].describe().round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(v2_features):
    df_v2[col].hist(bins=30, ax=axes[i], color="steelblue", edgecolor="white")
    axes[i].set_title(col, fontsize=10)
axes[-1].set_visible(False)
plt.suptitle("v2 Intensity Feature Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Feature Summary

Complete table of all engineered features used in the model.

In [ ]:
feature_table = pd.DataFrame([
    ("age_ordinal", "Numeric", "Age bracket encoded as ordinal 0-9"),
    ("time_in_hospital", "Numeric", "Days spent in hospital (1-14)"),
    ("num_lab_procedures", "Numeric", "Number of lab tests performed"),
    ("num_procedures", "Numeric", "Number of non-lab procedures"),
    ("num_medications", "Numeric", "Number of distinct medications administered"),
    ("number_diagnoses", "Numeric", "Number of diagnoses on record"),
    ("num_med_changes", "Numeric", "Count of medications with dosage Up or Down"),
    ("num_meds_active", "Numeric", "Count of medications currently prescribed"),
    ("total_prior_visits", "Numeric", "Sum of inpatient + outpatient + emergency visits"),
    ("insulin_changed", "Binary", "Whether insulin dosage was modified"),
    ("high_utilizer", "Binary", "1 if total_prior_visits >= 5"),
    ("medical_specialty_missing", "Binary", "1 if medical specialty not recorded"),
    ("lab_procedure_intensity", "Numeric (v2)", "num_lab_procedures / (time_in_hospital + 1)"),
    ("procedure_intensity", "Numeric (v2)", "num_procedures / (time_in_hospital + 1)"),
    ("medication_intensity", "Numeric (v2)", "num_medications / (time_in_hospital + 1)"),
    ("diagnosis_complexity", "Numeric (v2)", "number_diagnoses / (time_in_hospital + 1)"),
    ("med_change_ratio", "Numeric (v2)", "num_med_changes / (num_meds_active + 1)"),
    ("race", "Categorical", "Patient race (one-hot encoded)"),
    ("gender", "Categorical", "Patient gender (one-hot encoded)"),
    ("diag1_category", "Categorical", "Primary diagnosis ICD-9 category"),
    ("diag2_category", "Categorical", "Secondary diagnosis ICD-9 category"),
    ("diag3_category", "Categorical", "Tertiary diagnosis ICD-9 category"),
    ("admission_type_id", "Categorical", "Type of admission (emergency, urgent, elective, etc.)"),
], columns=["Feature", "Type", "Description"])

print(f"Total engineered features: {len(feature_table)}")
feature_table

## 8. Feature Importance Preview

Quick random forest to get a baseline sense of which features matter most.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from src.data.preprocessor import clean_data, build_preprocessing_pipeline

df_eng = run_feature_engineering(df.copy())
df_clean = clean_data(df_eng)

numeric_features = [
    "age_ordinal", "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_diagnoses", "num_med_changes", "num_meds_active",
    "total_prior_visits", "insulin_changed", "high_utilizer", "medical_specialty_missing",
]
categorical_features = ["race", "gender", "diag1_category", "diag2_category", "diag3_category", "admission_type_id"]

num_avail = [c for c in numeric_features if c in df_clean.columns]
cat_avail = [c for c in categorical_features if c in df_clean.columns]

preprocessor = build_preprocessing_pipeline(num_avail, cat_avail)
X = preprocessor.fit_transform(df_clean[num_avail + cat_avail])
y = df_clean["readmitted"].values

feature_names = num_avail + list(
    preprocessor.named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(cat_avail)
)

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importances.head(20).sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 20 Features by Random Forest Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()